In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, BatteryMaterials, ElectricVehicleBatteries_number, ElectricVehicleBatteries_weight
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.battery_preprocessing import get_preprocessing_data_evbattery

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials

In [ ]:
path_base

In [ ]:
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)


In [ ]:
prep_data = get_preprocessing_data_evbattery(path_base=path_base, scenario = "SSP2_M_CP", year_start=1970, year_out=2100)
sec_evbattery = Sector("ev_batteries", prep_data, check_coordinates=False)

In [ ]:
prep_data

In [ ]:
time_start = 2020
complete_timeline = prism.Timeline(time_start, 2030, 1)
simulation_timeline = prism.Timeline(2020, 2030, 1)

factory = ModelFactory(
    [vhc_sector, sec_evbattery], complete_timeline
    ).add(GenericStocks, ["vehicles"]
    ).add(GenericMaterials, ["vehicles"]
    ).add(ElectricVehicleBatteries, ["ev_batteries"], input_sources={
    "stock_by_cohort": "vehicles",
    "inflow": "vehicles",
    "outflow_by_cohort": "vehicles"}
    )
model2 = factory.finish()

warnings.filterwarnings("ignore")
model2.simulate(simulation_timeline)

list(model2.vehicles)

In [ ]:
model2.stock_battery_kg

In [ ]:
# da = model.inflow_battery.sum(["Region", "Type"])
da = model.stock_battery.sum(["Region", "Type", "Cohort"])

In [ ]:
da

In [ ]:

da_plot = da.sel(Time=slice(2000,2060))
fig, ax = plt.subplots()
for b in da_plot.battery.values:
    plt.plot(da_plot.Time, da_plot.sel(battery=b), label=b)
plt.legend()

In [ ]:
vhc_sector

In [ ]:
time_start = 2020
complete_timeline = prism.Timeline(time_start, 2030, 1)
simulation_timeline = prism.Timeline(2020, 2030, 1)

factory = ModelFactory(
    vhc_sector, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(BatteryMaterials
    )
model = factory.finish()

warnings.filterwarnings("ignore")
model.simulate(simulation_timeline)

list(model.vehicles)

In [ ]:
model.battery_weights
# model.battery_materials
# model.battery_shares
# model.battery_weights.sel(Type="Medium Freight Trucks - BEV")

# Batteries